In [1]:
# Imports
import os
import pandas as pd
from PIL import Image

In [2]:
# --- 1, 2, & 3. Scan Images, Extract Resolutions, and Calculate Aspect Ratio ---
dataset_path = "../dataset"
image_data = []

if os.path.exists(dataset_path):
    for class_name in sorted(os.listdir(dataset_path)):
        class_dir = os.path.join(dataset_path, class_name)
        if os.path.isdir(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(".jpg"):
                    file_path = os.path.join(class_dir, f)
                    try:
                        with Image.open(file_path) as img:
                            width, height = img.size
                            aspect_ratio = width / height if height > 0 else 0
                            image_data.append({
                                "Class": class_name,
                                "File_Path": file_path,
                                "Width": width,
                                "Height": height,
                                "Aspect_Ratio": aspect_ratio
                            })
                    except Exception as e:
                        print(f"Error reading file {file_path}: {e}")

    df_res = pd.DataFrame(image_data)
    print(f"Total scanned images for resolution analysis: {len(df_res)}")
else:
    print(f"Error: {dataset_path} does not exist")

Total scanned images for resolution analysis: 119338


In [3]:
# --- 4. Analyze Overall Resolution Distribution ---
if 'df_res' in locals() and not df_res.empty:
    print("--- Overall Resolution Statistics ---")
    print(df_res[["Width", "Height", "Aspect_Ratio"]].describe())

--- Overall Resolution Statistics ---
               Width         Height   Aspect_Ratio
count  119338.000000  119338.000000  119338.000000
mean      225.691666     219.418123       1.031076
std        74.778892      12.622641       0.338807
min       103.000000     121.000000       0.473214
25%       224.000000     224.000000       1.000000
50%       224.000000     224.000000       1.000000
75%       224.000000     224.000000       1.000000
max       473.000000     328.000000       2.111607


In [4]:
# --- 5. Per-Class Resolution Analysis ---
if 'df_res' in locals() and not df_res.empty:
    print("\n--- Per-Class Average Resolution ---")
    class_res_summary = df_res.groupby("Class")[["Width", "Height"]].mean().reset_index()
    display(class_res_summary.head(10))


--- Per-Class Average Resolution ---


,Class,Width,Height
0,A,201.052708,220.573125
1,B,284.381875,216.132708
2,C,198.183333,221.243542
3,D,196.850000,221.764792
4,E,198.657917,220.180417
5,F,278.178750,218.313333
6,G,271.925000,219.960417
7,H,204.014792,220.946458
8,I,188.454375,223.481667
9,J,272.154167,220.683125


In [5]:
# --- 6. Find Extremely Small / Large Images ---
if 'df_res' in locals() and not df_res.empty:
    # Define pixel thresholds for extreme sizes
    min_size_threshold = 100
    max_size_threshold = 3000

    extreme_size_images = df_res[
        (df_res["Width"] < min_size_threshold) | (df_res["Height"] < min_size_threshold) |
        (df_res["Width"] > max_size_threshold) | (df_res["Height"] > max_size_threshold)
    ]

    print(f"Number of images with extreme sizes: {len(extreme_size_images)}")
    if not extreme_size_images.empty:
        display(extreme_size_images.head())

Number of images with extreme sizes: 0


In [6]:
# --- 7. Find Unusual Aspect Ratios ---
if 'df_res' in locals() and not df_res.empty:
    # Define bounds for unusual aspect ratios (e.g., extremely long or wide)
    low_ar = 0.5
    high_ar = 2.0

    unusual_ar_images = df_res[
        (df_res["Aspect_Ratio"] < low_ar) | (df_res["Aspect_Ratio"] > high_ar)
    ]

    print(f"Number of images with unusual aspect ratios: {len(unusual_ar_images)}")
    if not unusual_ar_images.empty:
        display(unusual_ar_images.head())

Number of images with unusual aspect ratios: 18170


,Class,File_Path,Width,Height,Aspect_Ratio
4,A,../dataset/A/A_0_8891.jpg,106,224,0.473214
5,A,../dataset/A/A_0_933.jpg,106,224,0.473214
11,A,../dataset/A/A_0_6510.jpg,106,224,0.473214
14,A,../dataset/A/A_0_4409.jpg,106,224,0.473214
20,A,../dataset/A/A_0_6525.jpg,106,224,0.473214


In [7]:
# --- 8. Report Results ---
if 'df_res' in locals() and not df_res.empty:
    print("\n" + "="*45)
    print("RESOLUTION ANALYSIS REPORT")
    print("="*45)
    print(f"Total Images Analyzed      : {len(df_res)}")
    print(f"Unique Resolutions Found   : {df_res[['Width', 'Height']].drop_duplicates().shape[0]}")
    print(f"Extreme Size Images Count  : {len(extreme_size_images)}")
    print(f"Unusual Aspect Ratio Count : {len(unusual_ar_images)}")
    print("="*45)


RESOLUTION ANALYSIS REPORT
Total Images Analyzed      : 119338
Unique Resolutions Found   : 160
Extreme Size Images Count  : 0
Unusual Aspect Ratio Count : 18170
